# 车辆让行预测——lora微调

### Step1导入相关包

In [ ]:
import transformers
from transformers import (
    Qwen3VLForConditionalGeneration, 
    Qwen3VLProcessor, 
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments
)
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import json
import torch
from typing import Dict
import os
from PIL import Image # 新增：用于加载图片

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

### step2 加载数据集合

In [ ]:
with open('../dataset/train_dataset.json', 'r', encoding='utf-8') as f: 
    data = json.load(f)

print(data[0])

# 转换为Dataset格式
dataset = Dataset.from_list(data)
print(dataset)


### step3 加载模型

In [ ]:
#4-bit量化配置（节省显存）
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# bnb_config = BitsAndBytesConfig(
#     load_in_8bit=True,  # 改为 8-bit
#     llm_int8_threshold=6.0,  # 8-bit 量化阈值
#     llm_int8_has_fp16_weight=False,  # 不使用 fp16 权重
# )

In [ ]:
model_path="../model/Qwen3.5-4B-CRAFT-stage1"

model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,

)
model = prepare_model_for_kbit_training(model)

### Step4 添加图片数据

In [ ]:
import re 
IMAGE_ROOT_DIR = "../../Domain-Informed_Prompting/image" 
MAX_LENGTH=2048

def process_func(example):
    
    location_id_pattern = re.compile(r"Location ID:\s*(\d+)")
    match = location_id_pattern.search(example['input'])
    if not match:
        raise ValueError(f"未找到 Location ID: {example['input'][:50]}...")
    loc_id = match.group(1)
    
    image_path = os.path.join(IMAGE_ROOT_DIR, f"Slide{loc_id}.png")
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"图片不存在: {image_path}")

    # 直接返回原始数据 + 路径，不做任何 tokenize
    return {
        "image_path": image_path,
        "raw_instruction": example['instruction'],
        "raw_input": example['input'],
        "raw_output": example['output']
    }

tokenized_dataset = dataset.map(
    process_func, 
    remove_columns=dataset.column_names,
    load_from_cache_file=False,
)

In [ ]:
tokenized_dataset

### Step5划分训练和测试集

In [ ]:
split_dataset = tokenized_dataset.train_test_split(
    test_size=0.1,  # 测试集比例，按需调整
    seed=42,  # 固定随机种子，保证实验可复现
    shuffle=True
)

# 提取划分后的训练集和测试集
train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]
print(train_dataset)
print(test_dataset)


### step 6设置训练参数

lora参数

In [ ]:
lora_config = LoraConfig(
    r=8,  # LoRA秩（越大表达能力越强，显存消耗越高）
    lora_alpha=16,  # 缩放因子
    target_modules=[  # QWen3的注意力层模块名（需根据模型结构调整）
        "q_proj", "k_proj", "v_proj", "o_proj",  # 注意力Q/K/V/O投影层
        "gate_proj", "up_proj", "down_proj"  # FFN层（可选，增强微调效果）
    ],
    lora_dropout=0.05,
    fan_in_fan_out=False,
    bias="none",  # 不微调偏置参数
    task_type="CAUSAL_LM"  # 因果语言模型任务
)

# 将LoRA应用到模型
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
for name, parameter in model.named_parameters():
    print(name," 模型的激活状态 ",parameter.requires_grad)

训练参数

In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    
    # Shift
    labels = labels[:, 1:]
    preds = preds[:, :-1]
    
    # 过滤 -100
    true_predictions = [
        [p for (p, l) in zip(pred, gold_label) if l != -100]
        for pred, gold_label in zip(preds, labels)
    ]
    true_labels = [
        [l for (p, l) in zip(pred, gold_label) if l != -100]
        for pred, gold_label in zip(preds, labels)
    ]
    
    ID2LABEL = {
        tokenizer.convert_tokens_to_ids("1"): 1,  # yield → 正类
        tokenizer.convert_tokens_to_ids("0"): 0  # no_yield → 负类
    }


    y_pred = np.array([ID2LABEL[p[-1]] for p in true_predictions])
    y_true = np.array([ID2LABEL[l[-1]] for l in true_labels])
    


    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred)
    }
    return metrics

def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)



### Step7 训练时动态分词

In [ ]:
# 加载模型和tokenizer
tokenizer_path="../model/Qwen3.5-4B"
processor = Qwen3VLProcessor.from_pretrained(
    tokenizer_path,
    trust_remote_code=True
)

print(processor)

知识注入（RAG）

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

# 加载 embedding 模型
Embeddingmodel = SentenceTransformer('../model/Qwen3-Embedding-0.6B')

# 初始化 Chroma 客户端
chroma_client = chromadb.PersistentClient(path="../dataset/driver_yield_vector_db")

collection = chroma_client.get_collection(name="driver_yield_knowledge-28")


def retrieve_knowledge(query_text, top_k=8):
    """
    根据 input 文本检索最相关的领域知识
    """
    query_embedding = Embeddingmodel.encode(query_text).tolist()

    results = collection.query(query_embeddings=[query_embedding],n_results=top_k)

    retrieved_docs = results["documents"][0]  # list[str]

    # 拼接成一个字符串
    knowledge_text = "\n".join(retrieved_docs)

    return knowledge_text



In [ ]:
from typing import Any, List, Mapping
import transformers


def data_collator(features: List[Mapping[str, Any]]) -> Mapping[str, Any]:


    image_paths = [f["image_path"] for f in features]
    images = [Image.open(path).convert("RGB") for path in image_paths]
    
    

    # 2. 批量构建 Prompt
    texts = []
    for feature in features:
        
        retrieved_knowledge = retrieve_knowledge(feature['raw_input'], top_k=8)

        enhanced_instruction = (
        feature['raw_instruction']
        + "\n\n### Relevant Domain Knowledge:\n"
        + retrieved_knowledge
        )

        prompt = (
            f"<|im_start|>system\n{enhanced_instruction}<|im_end|>\n"
            f"<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|>\n{feature['raw_input']}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )
        full_text = prompt + feature['raw_output']
        texts.append(full_text)


    inputs = processor(
        text=texts,
        images=images,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )


    labels = inputs["input_ids"].clone()

    labels[:, :-7] = -100
    labels[inputs["attention_mask"] == 0] = -100

    inputs["labels"] = labels

    return inputs


In [ ]:
training_args = TrainingArguments(
    output_dir="./checkpoint/Qwen3.5-4B-CRAFT",
    per_device_train_batch_size=2,  # VL模型通常显存占用大，建议 batch_size 小一点
    gradient_accumulation_steps=8, # 增大梯度累积
    learning_rate=2e-5,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=50,
    bf16=True, # 确保显卡支持 (Ampere 及以上)
    optim="paged_adamw_8bit",
    report_to="none",
    eval_strategy="steps",
    eval_steps=10,
    per_device_eval_batch_size=1,
    load_best_model_at_end=True,
    remove_unused_columns=False, 
)


trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    data_collator=data_collator, # 使用自定义的 collator
    args=training_args,
)

### Step 7 开始训练

In [ ]:
print(model.hf_device_map)
trainer.train()